In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from vtk import vtkXMLImageDataReader
from vtk.util.numpy_support import vtk_to_numpy

from glob import glob
import os.path

# import sys
# sys.path.append("../../scripts")

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [53]:
class VTKReader:
    def __init__(self, content_dir):
        self.content_dir = content_dir
        self.phi = []

    def _detect_files(self):
        self.files = glob(os.path.join(self.content_dir, 'out_*[0-9].vti'))
        print('Detected {} vti files'.format(len(self.files)))
        self.files = sorted([(len(file_name), file_name) for file_name in self.files])
        self.files = [file_name for (_, file_name) in self.files]

    def _read_array(self, output, name):
        array_vtk = output.GetCellData().GetArray(name)
        array_np = vtk_to_numpy(array_vtk).reshape(self.dims - 1, order='F')
        return array_np

    def load(self):
        self._detect_files()
        for file in self.files:
            reader = vtkXMLImageDataReader()
            reader.SetFileName(file)
            reader.Update()
            output = reader.GetOutput()
            self.dims = np.array(output.GetDimensions())
            self.phi.append(self._read_array(output, 'Phase_field'))
        self.phi = np.stack(self.phi)
        print('Loading done!')

In [54]:
vtk_reader = VTKReader('/home/andrey/Desktop/kiam/calculation_examples/test_33_adapt_none/OUTPUT/vtk')
vtk_reader.load()
phi_first = vtk_reader.phi

Detected 11 vti files
Loading done!


In [55]:
vtk_reader = VTKReader('/home/andrey/Desktop/kiam/calculation_examples/test_33_adapt_phi_times/OUTPUT/vtk')
vtk_reader.load()
phi_second = vtk_reader.phi

Detected 11 vti files
Loading done!


In [59]:
def norm_uniform(first, second):
    return np.abs(first - second).max(axis=(1, 2, 3))

In [60]:
norm_uniform(phi_first, phi_second)

array([0.        , 0.0025144 , 0.01715739, 0.01511311, 0.27737532,
       0.36573993, 0.06069797, 0.06091748, 0.39507523, 0.41477431,
       0.44348386])